# History of Information Retrieval

You are starting the first notebook in this retrieval section.

You will learn why retrieval comes before generation.

You will use one tiny corpus, run each retrieval method, and read each math idea in plain English.

**A quick note before you start:** if some of the math feels unfamiliar at first, that is completely okay. You do not need to memorise every formula. The goal is to understand the concepts and build intuition for how each method works. If you get stuck on a specific piece of math, you can always paste it into ChatGPT and ask it to explain step by step. The key is understanding *what* each method does and *why* it matters, not memorising equations.

## Why Retrieval Matters

You can think about one simple question.

A user asks: **"Tell me about young dogs."**

If you only use exact keywords, you can miss text that says **"puppies"**.
If you only use semantic search, you can miss exact names, IDs, and strict terms.

That is why you first retrieve evidence, then generate an answer.

## What You Will Build

1. Levenshtein distance for typo tolerance.
2. Jaccard similarity for word overlap.
3. TFIDF for keyword weighting.
4. Word vectors for semantic search.
5. A toy neural ranker for learned relevance.
6. Hybrid retrieval that combines signals.


In [17]:
# DEPENDENCY: pip install -q --upgrade numpy
# (packages should be pre-installed in venv before running this script)

Note: you may need to restart the kernel to use updated packages.


In [18]:
import numpy as np
from scipy.spatial.distance import cosine
import re
from collections import Counter

## Your Toy Corpus

You will use one tiny corpus in every section.

This lets you compare methods fairly.


In [19]:
# Tiny corpus about animals and simple descriptions
documents = [
    "The cat sat on the mat",
    "The dog chased the cat in the garden",
    "A kitten is a young cat",
    "Dogs are loyal and friendly pets",
    "The mat was soft and warm",
    "Puppies and kittens are adorable baby animals",
    "The garden was full of flowers and birds",
    "Cats and dogs are the most popular pets worldwide",
]

def tokenize(text: str) -> list[str]:
    """Simple lowercase tokenizer."""
    return re.findall(r'\b\w+\b', text.lower())

def exact_match_search(query: str, docs: list[str]) -> list[str]:
    """Very naive baseline: only returns docs containing the exact string."""
    q = query.lower().strip()
    return [doc for doc in docs if q in doc.lower()]

print(f"Corpus size: {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc}")

print("\nQuick baseline check (exact match only):")
for query in ["puppy", "kitten", "young cat"]:
    matches = exact_match_search(query, documents)
    print(f"  Query '{query}' -> {len(matches)} exact match(es)")


Corpus size: 8 documents
  [0] The cat sat on the mat
  [1] The dog chased the cat in the garden
  [2] A kitten is a young cat
  [3] Dogs are loyal and friendly pets
  [4] The mat was soft and warm
  [5] Puppies and kittens are adorable baby animals
  [6] The garden was full of flowers and birds
  [7] Cats and dogs are the most popular pets worldwide

Quick baseline check (exact match only):
  Query 'puppy' -> 0 exact match(es)
  Query 'kitten' -> 2 exact match(es)
  Query 'young cat' -> 1 exact match(es)


## 1. Levenshtein Distance

You use Levenshtein distance to handle spelling mistakes.

You fill a dynamic programming table.

Math in plain English:
1. For each cell, check the value above and add one edit.
2. Check the value to the left and add one edit.
3. Check the diagonal value and add zero if characters match, otherwise add one.
4. Choose the smallest value.

You now have the minimum number of edits required to transform one string into another.


In [ ]:
'testa', 'test'
# Distance = 1

('testa', 'testa')

In [21]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """Compute the Levenshtein (edit) distance between two strings using dynamic programming."""
    m, n = len(s1), len(s2)
    
    # Create a matrix of size (m+1) x (n+1)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    # Base cases: transforming empty string to/from a prefix
    for i in range(m + 1):
        dp[i][0] = i  # delete all chars from s1
    for j in range(n + 1):
        dp[0][j] = j  # insert all chars into s1
    
    # Fill the matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # characters match, no edit needed
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # deletion
                    dp[i][j - 1],      # insertion
                    dp[i - 1][j - 1]   # substitution
                )
    
    return dp[m][n]

# Let's see it in action
pairs = [("cat", "cat"), ("cat", "cta"), ("cat", "cats"), ("kitten", "sitting"), ("dog", "cat")]

for a, b in pairs:
    dist = levenshtein_distance(a, b)
    print(f"  '{a}' -> '{b}' = {dist} edit(s)")

  'cat' -> 'cat' = 0 edit(s)
  'cat' -> 'cta' = 2 edit(s)
  'cat' -> 'cats' = 1 edit(s)
  'kitten' -> 'sitting' = 3 edit(s)
  'dog' -> 'cat' = 3 edit(s)


In [22]:
def levenshtein_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Search documents by finding the closest Levenshtein match for each query word.
    
    For each word in the query, we find the minimum edit distance to any word
    in each document. Lower total distance = better match.
    """
    query_words = tokenize(query)
    scores = []
    
    for idx, doc in enumerate(documents):
        doc_words = tokenize(doc)
        total_distance = 0
        
        for qw in query_words:
            # Find the closest word in the document
            min_dist = min(levenshtein_distance(qw, dw) for dw in doc_words)
            total_distance += min_dist
        
        # Normalize by number of query words (lower is better)
        avg_dist = total_distance / len(query_words)
        scores.append((idx, avg_dist, doc))
    
    # Sort by distance (ascending — lower is better)
    scores.sort(key=lambda x: x[1])
    return scores[:top_k]

# Test: exact match
print("Query: 'cat mat'")
for idx, dist, doc in levenshtein_search("cat mat", documents):
    print(f"  [{idx}] dist={dist:.2f} | {doc}")

# Test: typo in query
print("\nQuery: 'cta' (typo for 'cat')")
for idx, dist, doc in levenshtein_search("cta", documents):
    print(f"  [{idx}] dist={dist:.2f} | {doc}")

Query: 'cat mat'
  [0] dist=0.00 | The cat sat on the mat
  [1] dist=0.50 | The dog chased the cat in the garden
  [2] dist=0.50 | A kitten is a young cat

Query: 'cta' (typo for 'cat')
  [0] dist=2.00 | The cat sat on the mat
  [1] dist=2.00 | The dog chased the cat in the garden
  [2] dist=2.00 | A kitten is a young cat


You now learned a key limit.

Levenshtein helps with typos, but it does not understand meaning.
The words "dog" and "puppy" are close in meaning, but not in character edits.


## 2. Jaccard Similarity

You can measure overlap between query words and document words.

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Math in plain English:
1. Put query words into set A.
2. Put document words into set B.
3. Count shared words with intersection.
4. Count total unique words with union.
5. Divide shared count by total count.

A score near one means strong overlap.
A score near zero means weak overlap.


In [23]:
def jaccard_similarity(set_a: set, set_b: set) -> float:
    """Compute Jaccard similarity between two sets."""
    intersection = set_a & set_b
    union = set_a | set_b
    if not union:
        return 0.0
    return len(intersection) / len(union)

def jaccard_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Rank documents by Jaccard similarity with the query."""
    query_set = set(tokenize(query))
    scores = []
    
    for idx, doc in enumerate(documents):
        doc_set = set(tokenize(doc))
        sim = jaccard_similarity(query_set, doc_set)
        scores.append((idx, sim, doc))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Demonstrate with individual set comparisons first
q = set(tokenize("cat on the mat"))
d0 = set(tokenize(documents[0]))
d1 = set(tokenize(documents[1]))

print(f"Query words:  {q}")
print(f"Doc 0 words:  {d0}")
print(f"Intersection: {q & d0}")
print(f"Union:        {q | d0}")
print(f"Jaccard:      {jaccard_similarity(q, d0):.3f}")
print()
print(f"Doc 1 words:  {d1}")
print(f"Intersection: {q & d1}")
print(f"Jaccard:      {jaccard_similarity(q, d1):.3f}")

Query words:  {'the', 'on', 'cat', 'mat'}
Doc 0 words:  {'sat', 'mat', 'the', 'cat', 'on'}
Intersection: {'the', 'on', 'cat', 'mat'}
Union:        {'sat', 'mat', 'the', 'cat', 'on'}
Jaccard:      0.800

Doc 1 words:  {'the', 'garden', 'cat', 'chased', 'dog', 'in'}
Intersection: {'the', 'cat'}
Jaccard:      0.250


In [24]:
# Full search
print("Query: 'cat on the mat'")
for idx, score, doc in jaccard_search("cat on the mat", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'loyal friendly pets'")
for idx, score, doc in jaccard_search("loyal friendly pets", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

Query: 'cat on the mat'
  [0] score=0.800 | The cat sat on the mat
  [1] score=0.250 | The dog chased the cat in the garden
  [4] score=0.250 | The mat was soft and warm

Query: 'loyal friendly pets'
  [3] score=0.500 | Dogs are loyal and friendly pets
  [7] score=0.091 | Cats and dogs are the most popular pets worldwide
  [0] score=0.000 | The cat sat on the mat


You now learned another limit.

Jaccard is simple and fast.
It treats every word equally and ignores word frequency.


## 3. TFIDF

You now improve keyword retrieval by weighting words.

Core formula:

$$\text{TFIDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Smoothed IDF used in this notebook:

$$\text{IDF}(t) = \log\left(\frac{N}{\text{df}(t)+1}\right)$$

Math in plain English:
1. TF tells you how often a term appears in one document.
2. df tells you in how many documents that term appears.
3. N is the number of documents in your corpus.
4. Common terms get lower IDF.
5. Rare terms get higher IDF.
6. You multiply TF and IDF to get each term weight.

You then compare query and document vectors with cosine similarity.


In [1]:
import math

number_of_documents = 1000
rare_term_count = 2
frequent_word_count = 950

print(math.log(number_of_documents / (rare_term_count + 1)))
print(math.log(number_of_documents / (frequent_word_count + 1)))

# Now we'll calculate the Term Frequency:
rare_term_count_frequency = rare_term_count / number_of_documents
frequent_word_count_frequency = frequent_word_count / number_of_documents

print(rare_term_count_frequency)
print(frequent_word_count_frequency)

# Now we'll calculate the TF-IDF:
rare_term_tfidf = rare_term_count_frequency * math.log(number_of_documents / (rare_term_count + 1))
frequenct_word_tfidf = frequent_word_count_frequency * math.log(number_of_documents / (frequent_word_count + 1))

print('tfidf for rare term:', rare_term_tfidf)
print('tfidf for frequent term:', frequenct_word_tfidf)

5.809142990314028
0.05024121643674665
0.002
0.95
tfidf for rare term: 0.011618285980628055
tfidf for frequent term: 0.04772915561490931


In [2]:
class TFIDFRetriever:
    """TF-IDF retrieval from scratch using numpy."""
    
    def __init__(self):
        self.vocab = []         # list of all unique words
        self.word_to_idx = {}   # word -> index in vocab
        self.idf = None         # IDF scores
        self.tfidf_matrix = None  # documents x vocab TF-IDF matrix
    
    def fit(self, documents: list[str]):
        """Build the TF-IDF matrix."""
        self.documents = documents
        tokenized = [tokenize(doc) for doc in documents]
        
        # Build vocabulary
        all_words = set()
        for tokens in tokenized:
            all_words.update(tokens)
        self.vocab = sorted(all_words)
        self.word_to_idx = {w: i for i, w in enumerate(self.vocab)}
        
        n_docs = len(documents)
        n_terms = len(self.vocab)
        
        # Compute Term Frequency matrix
        tf = np.zeros((n_docs, n_terms))
        for doc_idx, tokens in enumerate(tokenized):
            counts = Counter(tokens)
            for word, count in counts.items():
                tf[doc_idx, self.word_to_idx[word]] = count
        
        # Normalize TF by document length
        doc_lengths = tf.sum(axis=1, keepdims=True)
        doc_lengths[doc_lengths == 0] = 1  # avoid division by zero
        tf = tf / doc_lengths
        
        # Compute IDF
        doc_freq = (tf > 0).sum(axis=0)  # number of docs containing each term
        self.idf = np.log(n_docs / (doc_freq + 1)) + 1  # smoothed IDF
        
        # TF-IDF = TF * IDF
        self.tfidf_matrix = tf * self.idf
        
        print(f"Vocabulary size: {n_terms}")
        print(f"TF-IDF matrix shape: {self.tfidf_matrix.shape} (docs x terms)")
    
    def _query_vector(self, query: str) -> np.ndarray:
        """Convert a query string into a TF-IDF vector."""
        tokens = tokenize(query)
        vec = np.zeros(len(self.vocab))
        counts = Counter(tokens)
        for word, count in counts.items():
            if word in self.word_to_idx:
                vec[self.word_to_idx[word]] = count
        # Normalize TF
        total = vec.sum()
        if total > 0:
            vec = vec / total
        # Apply IDF
        vec = vec * self.idf
        return vec
    
    def search(self, query: str, top_k: int = 3) -> list[tuple[int, float, str]]:
        """Search using cosine similarity between query and document TF-IDF vectors."""
        q_vec = self._query_vector(query)
        
        scores = []
        for idx in range(len(self.documents)):
            d_vec = self.tfidf_matrix[idx]
            # Cosine similarity (scipy.cosine returns distance, so 1 - distance)
            if np.linalg.norm(q_vec) == 0 or np.linalg.norm(d_vec) == 0:
                sim = 0.0
            else:
                sim = 1 - cosine(q_vec, d_vec)
            scores.append((idx, sim, self.documents[idx]))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

tfidf = TFIDFRetriever()
tfidf.fit(documents)

NameError: name 'np' is not defined

In [3]:
print("Query: 'cat mat'")
for idx, score, doc in tfidf.search("cat mat"):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'loyal friendly pets'")
for idx, score, doc in tfidf.search("loyal friendly pets"):
    print(f"  [{idx}] score={score:.3f} | {doc}")

# Show that TF-IDF down-weights common words
print("\nIDF scores for selected words:")
for word in ["the", "cat", "kitten", "loyal", "adorable"]:
    if word in tfidf.word_to_idx:
        print(f"  '{word}': {tfidf.idf[tfidf.word_to_idx[word]]:.3f}")

Query: 'cat mat'


NameError: name 'tfidf' is not defined

### Cosine Similarity in Plain English

$$\cos(\theta) = \frac{q \cdot d}{\|q\|\|d\|}$$

Math in plain English:
1. The dot product checks how strongly important terms align.
2. The denominator scales by vector lengths.
3. This gives you direction based similarity, not raw length.
4. A score near one means high similarity.
5. A score near zero means low similarity.


In [64]:
# Toy cosine similarity — 2D space: [cat-ness, dog-ness]
# Each number = how much the doc is "about" that topic (0 to 1)

query_vector = np.array([1.0, 0.0])  # Query: "cats" only
document_all_cats = np.array([1.0, 0.0])  # Doc: 100% cats → same as query
document_all_dogs = np.array([0.0, 1.0])  # Doc: 100% dogs → orthogonal
document_half_and_half = np.array([1.0, 1.0])  # Doc: 50% cats, 50% dogs


def cosine_similarity(vector_a, vector_b):
    """cos(θ) = (a·b) / (||a|| × ||b||)"""
    # np.dot(a, b) = sum of element-wise products. For [1,0] and [1,1]: 1*1 + 0*1 = 1
    # Measures how much the vectors "align" — big when same direction, 0 when perpendicular
    dot_product = np.dot(vector_a, vector_b)

    # np.linalg.norm(x) = length of vector = sqrt(sum of squares). For [1,0]: sqrt(1) = 1
    # For [1,1]: sqrt(1+1) = 1.41. Divides by length so we compare direction, not magnitude
    length_of_a = np.linalg.norm(vector_a)
    length_of_b = np.linalg.norm(vector_b)

    if length_of_a == 0 or length_of_b == 0:
        return 0.0
    return dot_product / (length_of_a * length_of_b)


print("Query [1, 0] = 'cats only'\n")
print(f"  vs all-cats doc [1,0]:  cos = {cosine_similarity(query_vector, document_all_cats):.2f}  (identical)")
print(f"  vs all-dogs doc [0,1]:  cos = {cosine_similarity(query_vector, document_all_dogs):.2f}  (orthogonal)")
print(f"  vs half-half [1,1]:     cos = {cosine_similarity(query_vector, document_half_and_half):.2f}  (partial match)")
print("\nCosine ignores length — only direction matters.")


Query [1, 0] = 'cats only'

  vs all-cats doc [1,0]:  cos = 1.00  (identical)
  vs all-dogs doc [0,1]:  cos = 0.00  (orthogonal)
  vs half-half [1,1]:     cos = 0.71  (partial match)

Cosine ignores length — only direction matters.


You now learned why TFIDF is stronger than raw overlap, but still limited for synonyms.

## 4. Word Vectors

You now move from keyword overlap to semantic similarity.

Words with similar meaning can be close in vector space.

In this toy notebook, you use hand built vectors so you can inspect behavior directly.


In [65]:
# Hand-crafted word vectors to illustrate the concept.
# Dimensions roughly correspond to: [animal, pet, young, location, soft/warm, action, plant]
word_vectors = {
    "cat":      np.array([0.9, 0.8, 0.0, 0.0, 0.2, 0.0, 0.0]),
    "kitten":   np.array([0.9, 0.8, 0.9, 0.0, 0.3, 0.0, 0.0]),
    "dog":      np.array([0.9, 0.9, 0.0, 0.0, 0.1, 0.3, 0.0]),
    "puppy":    np.array([0.9, 0.9, 0.9, 0.0, 0.2, 0.3, 0.0]),
    "puppies":  np.array([0.9, 0.9, 0.9, 0.0, 0.2, 0.3, 0.0]),
    "kittens":  np.array([0.9, 0.8, 0.9, 0.0, 0.3, 0.0, 0.0]),
    "cats":     np.array([0.9, 0.8, 0.0, 0.0, 0.2, 0.0, 0.0]),
    "dogs":     np.array([0.9, 0.9, 0.0, 0.0, 0.1, 0.3, 0.0]),
    "pets":     np.array([0.7, 1.0, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "animals":  np.array([1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "baby":     np.array([0.3, 0.2, 1.0, 0.0, 0.4, 0.0, 0.0]),
    "young":    np.array([0.2, 0.1, 0.9, 0.0, 0.1, 0.0, 0.0]),
    "adorable": np.array([0.4, 0.5, 0.6, 0.0, 0.5, 0.0, 0.0]),
    "loyal":    np.array([0.3, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "friendly": np.array([0.3, 0.5, 0.0, 0.0, 0.2, 0.0, 0.0]),
    "sat":      np.array([0.0, 0.0, 0.0, 0.1, 0.1, 0.8, 0.0]),
    "chased":   np.array([0.0, 0.0, 0.0, 0.1, 0.0, 0.9, 0.0]),
    "mat":      np.array([0.0, 0.0, 0.0, 0.6, 0.8, 0.0, 0.0]),
    "garden":   np.array([0.0, 0.0, 0.0, 0.9, 0.2, 0.0, 0.7]),
    "flowers":  np.array([0.0, 0.0, 0.0, 0.5, 0.1, 0.0, 1.0]),
    "birds":    np.array([0.8, 0.1, 0.0, 0.4, 0.0, 0.3, 0.2]),
    "soft":     np.array([0.0, 0.0, 0.0, 0.1, 0.9, 0.0, 0.0]),
    "warm":     np.array([0.0, 0.0, 0.0, 0.1, 0.9, 0.0, 0.0]),
    "popular":  np.array([0.1, 0.3, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "worldwide":np.array([0.0, 0.1, 0.0, 0.2, 0.0, 0.0, 0.0]),
    "most":     np.array([0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "full":     np.array([0.0, 0.0, 0.0, 0.3, 0.0, 0.0, 0.2]),
}

# Show that similar words have similar vectors
print("Cosine similarity between word pairs:")
pairs = [("cat", "kitten"), ("cat", "dog"), ("cat", "garden"), ("puppy", "kitten"), ("mat", "soft")]
for w1, w2 in pairs:
    sim = 1 - cosine(word_vectors[w1], word_vectors[w2])
    print(f"  {w1:8s} <-> {w2:8s} = {sim:.3f}")

Cosine similarity between word pairs:
  cat      <-> kitten   = 0.807
  cat      <-> dog      = 0.968
  cat      <-> garden   = 0.028
  puppy    <-> kitten   = 0.978
  mat      <-> soft     = 0.861


In [66]:
def embed_text(text: str, vectors: dict) -> np.ndarray:
    """Create a document embedding by averaging the word vectors."""
    tokens = tokenize(text)
    vecs = [vectors[t] for t in tokens if t in vectors]
    if not vecs:
        return np.zeros(7)  # dimension of our vectors
    return np.mean(vecs, axis=0)

def embedding_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Search using average word-vector cosine similarity."""
    q_emb = embed_text(query, word_vectors)
    scores = []
    
    for idx, doc in enumerate(documents):
        d_emb = embed_text(doc, word_vectors)
        if np.linalg.norm(q_emb) == 0 or np.linalg.norm(d_emb) == 0:
            sim = 0.0
        else:
            sim = 1 - cosine(q_emb, d_emb)
        scores.append((idx, sim, doc))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Now we can search with SEMANTICALLY related words!
print("Query: 'puppy' (no document contains the word 'puppy' directly)")
for idx, score, doc in embedding_search("puppy", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'young animal'")
for idx, score, doc in embedding_search("young animal", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

Query: 'puppy' (no document contains the word 'puppy' directly)
  [5] score=0.981 | Puppies and kittens are adorable baby animals
  [2] score=0.978 | A kitten is a young cat
  [7] score=0.811 | Cats and dogs are the most popular pets worldwide

Query: 'young animal'
  [5] score=0.767 | Puppies and kittens are adorable baby animals
  [2] score=0.745 | A kitten is a young cat
  [7] score=0.229 | Cats and dogs are the most popular pets worldwide


You now learned what semantic search gives you.

It can retrieve related meaning even when exact words differ.
It can still lose detail because simple averaging removes word order.


## 5. Neural Ranking

You now use a toy neural ranker.

You give the model a query vector and a document vector.
The model outputs a relevance score between zero and one.
Training pushes relevant pairs up and irrelevant pairs down.

If you are new to neural training, focus on inputs and outputs first.


In [67]:
class ToyNeuralRanker:
    """A tiny neural network that scores query-document pairs.
    
    Architecture: concatenate query + doc embeddings -> hidden layer -> score
    This is a simplified cross-encoder pattern.
    """
    
    def __init__(self, input_dim: int = 7, hidden_dim: int = 8):
        np.random.seed(42)
        # Two embeddings concatenated = 2 * input_dim
        self.W1 = np.random.randn(2 * input_dim, hidden_dim) * 0.5
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, 1) * 0.5
        self.b2 = np.zeros(1)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, query_emb: np.ndarray, doc_emb: np.ndarray) -> float:
        """Score a query-document pair."""
        # Concatenate query and document embeddings
        x = np.concatenate([query_emb, doc_emb])
        # Hidden layer with ReLU
        h = self.relu(x @ self.W1 + self.b1)
        # Output score with sigmoid (0 to 1)
        score = self.sigmoid(h @ self.W2 + self.b2)
        return float(score[0])
    
    def train(self, examples: list[tuple[str, str, float]], epochs: int = 200, lr: float = 0.1):
        """Train on (query, document, relevance_label) triples."""
        print(f"Training on {len(examples)} examples for {epochs} epochs...")
        
        for epoch in range(epochs):
            total_loss = 0
            for query, doc, label in examples:
                q_emb = embed_text(query, word_vectors)
                d_emb = embed_text(doc, word_vectors)
                
                # Forward pass
                x = np.concatenate([q_emb, d_emb])
                h = self.relu(x @ self.W1 + self.b1)
                score = self.sigmoid(h @ self.W2 + self.b2)
                
                # Binary cross-entropy loss
                pred = float(score[0])
                loss = -(label * np.log(pred + 1e-8) + (1 - label) * np.log(1 - pred + 1e-8))
                total_loss += loss
                
                # Backprop (simplified gradient descent)
                d_score = pred - label  # gradient of BCE w.r.t. pre-sigmoid
                d_W2 = h.reshape(-1, 1) * d_score
                d_b2 = np.array([d_score])
                d_h = (self.W2.flatten() * d_score) * (h > 0).astype(float)
                d_W1 = x.reshape(-1, 1) @ d_h.reshape(1, -1)
                d_b1 = d_h
                
                # Update weights
                self.W2 -= lr * d_W2
                self.b2 -= lr * d_b2
                self.W1 -= lr * d_W1
                self.b1 -= lr * d_h
            
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}: loss = {total_loss / len(examples):.4f}")
    
    def search(self, query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
        """Rank documents for a query."""
        q_emb = embed_text(query, word_vectors)
        scores = []
        for idx, doc in enumerate(documents):
            d_emb = embed_text(doc, word_vectors)
            score = self.forward(q_emb, d_emb)
            scores.append((idx, score, doc))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

In [68]:
# Training data: (query, document, relevance)
# 1.0 = relevant, 0.0 = not relevant
training_data = [
    ("cat", "The cat sat on the mat", 1.0),
    ("cat", "The garden was full of flowers and birds", 0.0),
    ("dog", "Dogs are loyal and friendly pets", 1.0),
    ("dog", "The mat was soft and warm", 0.0),
    ("young animals", "Puppies and kittens are adorable baby animals", 1.0),
    ("young animals", "The garden was full of flowers and birds", 0.0),
    ("pets", "Cats and dogs are the most popular pets worldwide", 1.0),
    ("pets", "The mat was soft and warm", 0.0),
    ("kitten", "A kitten is a young cat", 1.0),
    ("kitten", "The dog chased the cat in the garden", 0.0),
]

ranker = ToyNeuralRanker()
ranker.train(training_data, epochs=120, lr=0.1)

Training on 10 examples for 120 epochs...
  Epoch 50: loss = 0.0317
  Epoch 100: loss = 0.0094


In [69]:
# Test the trained ranker
print("Query: 'cat'")
for idx, score, doc in ranker.search("cat", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'young animals'")
for idx, score, doc in ranker.search("young animals", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'puppy' (unseen in training!)")
for idx, score, doc in ranker.search("puppy", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

Query: 'cat'
  [5] score=1.000 | Puppies and kittens are adorable baby animals
  [2] score=1.000 | A kitten is a young cat
  [3] score=1.000 | Dogs are loyal and friendly pets

Query: 'young animals'
  [5] score=1.000 | Puppies and kittens are adorable baby animals
  [2] score=1.000 | A kitten is a young cat
  [3] score=1.000 | Dogs are loyal and friendly pets

Query: 'puppy' (unseen in training!)
  [5] score=0.998 | Puppies and kittens are adorable baby animals
  [2] score=0.994 | A kitten is a young cat
  [3] score=0.913 | Dogs are loyal and friendly pets


You now learned the neural ranking idea.

The model learns relevance patterns from examples.
Real production rankers scale this pattern with larger models and better training data.


## 6. Hybrid Retrieval

You now combine keyword and semantic retrieval.

You use Reciprocal Rank Fusion.

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

Math in plain English:
1. Each retriever gives every document a rank.
2. Better rank gives larger contribution.
3. You sum contributions across retrievers.
4. You sort by total fused score.

You now have one final ranking that uses both exact match and semantic meaning.


In [70]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[tuple[int, float, str]]],
    k: int = 60,
    top_k: int = 3
) -> list[tuple[int, float, str]]:
    """Combine multiple ranked lists using Reciprocal Rank Fusion."""
    rrf_scores = {}  # doc_idx -> score
    doc_texts = {}   # doc_idx -> text
    
    for ranked_list in ranked_lists:
        for rank, (doc_idx, _, doc_text) in enumerate(ranked_list):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + 1 / (k + rank + 1)
            doc_texts[doc_idx] = doc_text
    
    # Sort by combined RRF score
    results = [(idx, score, doc_texts[idx]) for idx, score in rrf_scores.items()]
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

def hybrid_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Hybrid retrieval combining TF-IDF (keyword) and embedding (semantic) search via RRF."""
    # Get rankings from both retrievers (return all docs so RRF has full picture)
    keyword_results = tfidf.search(query, top_k=len(documents))
    semantic_results = embedding_search(query, documents, top_k=len(documents))
    
    # Fuse with RRF
    return reciprocal_rank_fusion([keyword_results, semantic_results], top_k=top_k)

In [71]:
# Compare all three methods side by side
query = "puppy"

print(f"Query: '{query}'")
print(f"(Note: the word 'puppy' does NOT appear in any document)\n")

print("TF-IDF (keyword) results:")
kw_results = tfidf.search(query, top_k=3)
for idx, score, doc in kw_results:
    print(f"  [{idx}] score={score:.3f} | {doc}")
if not any(s > 0 for _, s, _ in kw_results):
    print("  (all scores are 0 — keyword search can't find 'puppy')")

print("\nEmbedding (semantic) results:")
for idx, score, doc in embedding_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nHybrid (TF-IDF + Embedding via RRF):")
for idx, score, doc in hybrid_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.4f} | {doc}")

Query: 'puppy'
(Note: the word 'puppy' does NOT appear in any document)

TF-IDF (keyword) results:
  [0] score=0.000 | The cat sat on the mat
  [1] score=0.000 | The dog chased the cat in the garden
  [2] score=0.000 | A kitten is a young cat
  (all scores are 0 — keyword search can't find 'puppy')

Embedding (semantic) results:
  [5] score=0.981 | Puppies and kittens are adorable baby animals
  [2] score=0.978 | A kitten is a young cat
  [7] score=0.811 | Cats and dogs are the most popular pets worldwide

Hybrid (TF-IDF + Embedding via RRF):
  [2] score=0.0320 | A kitten is a young cat
  [0] score=0.0315 | The cat sat on the mat
  [5] score=0.0315 | Puppies and kittens are adorable baby animals


In [72]:
# Another example where hybrid shines: a query mixing a specific term + semantics
query = "friendly kitten"

print(f"Query: '{query}'\n")

print("TF-IDF (keyword):")
for idx, score, doc in tfidf.search(query, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nEmbedding (semantic):")
for idx, score, doc in embedding_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nHybrid (RRF fusion):")
for idx, score, doc in hybrid_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.4f} | {doc}")

Query: 'friendly kitten'

TF-IDF (keyword):
  [3] score=0.346 | Dogs are loyal and friendly pets
  [2] score=0.258 | A kitten is a young cat
  [0] score=0.000 | The cat sat on the mat

Embedding (semantic):
  [2] score=0.985 | A kitten is a young cat
  [5] score=0.979 | Puppies and kittens are adorable baby animals
  [7] score=0.882 | Cats and dogs are the most popular pets worldwide

Hybrid (RRF fusion):
  [2] score=0.0325 | A kitten is a young cat
  [3] score=0.0320 | Dogs are loyal and friendly pets
  [5] score=0.0313 | Puppies and kittens are adorable baby animals


You now learned the practical lesson.

Hybrid retrieval usually works best because real queries need both exact matching and semantic matching.


## Exercises With Solutions

For each exercise, read the prompt and then run the solution code directly below it.


### Exercise 1

You will compare methods on one exact query and one semantic query.

Tasks:
1. Run the solution cell below.
2. Identify which method wins for "cat mat".
3. Identify which method wins for "puppy".
4. Explain why hybrid helps.


In [59]:
# Solution for Exercise 1

def compare_methods(query: str, top_k: int = 3):
    print(f"\n=== Query: '{query}' ===")

    print("TFIDF keyword results:")
    for idx, score, doc in tfidf.search(query, top_k=top_k):
        print(f"  [{idx}] score={score:.3f} | {doc}")

    print("\nEmbedding semantic results:")
    for idx, score, doc in embedding_search(query, documents, top_k=top_k):
        print(f"  [{idx}] score={score:.3f} | {doc}")

    print("\nHybrid RRF results:")
    for idx, score, doc in hybrid_search(query, documents, top_k=top_k):
        print(f"  [{idx}] score={score:.4f} | {doc}")

compare_methods("cat mat")
compare_methods("puppy")



=== Query: 'cat mat' ===
TFIDF keyword results:
  [0] score=0.523 | The cat sat on the mat
  [4] score=0.317 | The mat was soft and warm
  [1] score=0.177 | The dog chased the cat in the garden

Embedding semantic results:
  [0] score=0.910 | The cat sat on the mat
  [1] score=0.798 | The dog chased the cat in the garden
  [7] score=0.790 | Cats and dogs are the most popular pets worldwide

Hybrid RRF results:
  [0] score=0.0328 | The cat sat on the mat
  [1] score=0.0320 | The dog chased the cat in the garden
  [4] score=0.0311 | The mat was soft and warm

=== Query: 'puppy' ===
TFIDF keyword results:
  [0] score=0.000 | The cat sat on the mat
  [1] score=0.000 | The dog chased the cat in the garden
  [2] score=0.000 | A kitten is a young cat

Embedding semantic results:
  [5] score=0.981 | Puppies and kittens are adorable baby animals
  [2] score=0.978 | A kitten is a young cat
  [7] score=0.811 | Cats and dogs are the most popular pets worldwide

Hybrid RRF results:
  [2] score=0.0

### Exercise 2

You will tune hybrid weights and observe ranking changes.

Tasks:
1. Run the solution cell below.
2. Compare equal, keyword heavy, and semantic heavy settings.
3. Write one sentence on which setting you prefer for this corpus.


In [60]:
# Solution for Exercise 2

def weighted_rrf(
    ranked_lists: list[list[tuple[int, float, str]]],
    weights: list[float],
    k: int = 60,
    top_k: int = 3
) -> list[tuple[int, float, str]]:
    """RRF with per retriever weights."""
    if len(ranked_lists) != len(weights):
        raise ValueError("ranked_lists and weights must have the same length")

    fused_scores = {}
    doc_texts = {}

    for ranked_list, weight in zip(ranked_lists, weights):
        for rank, (doc_idx, _, doc_text) in enumerate(ranked_list):
            contribution = weight * (1 / (k + rank + 1))
            fused_scores[doc_idx] = fused_scores.get(doc_idx, 0.0) + contribution
            doc_texts[doc_idx] = doc_text

    results = [(idx, score, doc_texts[idx]) for idx, score in fused_scores.items()]
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

def weighted_hybrid_search(
    query: str,
    documents: list[str],
    keyword_weight: float = 1.0,
    semantic_weight: float = 1.0,
    top_k: int = 3
) -> list[tuple[int, float, str]]:
    """Hybrid search with adjustable keyword and semantic weights."""
    keyword_results = tfidf.search(query, top_k=len(documents))
    semantic_results = embedding_search(query, documents, top_k=len(documents))

    return weighted_rrf(
        ranked_lists=[keyword_results, semantic_results],
        weights=[keyword_weight, semantic_weight],
        top_k=top_k,
    )

query = "friendly kitten"
print(f"Query: '{query}'")

settings = [
    (1.0, 1.0, "Equal weights"),
    (2.0, 0.5, "Keyword heavy"),
    (0.5, 2.0, "Semantic heavy"),
]

for kw, sem, label in settings:
    print(f"\n{label} keyword={kw} semantic={sem}")
    for idx, score, doc in weighted_hybrid_search(query, documents, kw, sem, top_k=3):
        print(f"  [{idx}] score={score:.4f} | {doc}")


Query: 'friendly kitten'

Equal weights keyword=1.0 semantic=1.0
  [2] score=0.0325 | A kitten is a young cat
  [3] score=0.0320 | Dogs are loyal and friendly pets
  [5] score=0.0313 | Puppies and kittens are adorable baby animals

Keyword heavy keyword=2.0 semantic=0.5
  [3] score=0.0406 | Dogs are loyal and friendly pets
  [2] score=0.0405 | A kitten is a young cat
  [0] score=0.0393 | The cat sat on the mat

Semantic heavy keyword=0.5 semantic=2.0
  [2] score=0.0409 | A kitten is a young cat
  [5] score=0.0398 | Puppies and kittens are adorable baby animals
  [3] score=0.0394 | Dogs are loyal and friendly pets


### Exercise 3

You will build a three retriever hybrid.

Tasks:
1. Run the solution cell below.
2. Compare TFIDF only, embedding only, two way hybrid, and three way hybrid.
3. Note one ranking change you observe.


In [61]:
# Solution for Exercise 3

def three_way_hybrid(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Combine TFIDF, embedding, and Jaccard via RRF."""
    keyword_results = tfidf.search(query, top_k=len(documents))
    semantic_results = embedding_search(query, documents, top_k=len(documents))
    jaccard_results = jaccard_search(query, documents, top_k=len(documents))

    return reciprocal_rank_fusion(
        [keyword_results, semantic_results, jaccard_results],
        top_k=top_k,
    )

query = "adorable baby cat"
print(f"Query: '{query}'")

print("\nTFIDF only:")
for idx, score, doc in tfidf.search(query, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nEmbedding only:")
for idx, score, doc in embedding_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nTwo way hybrid TFIDF plus embedding:")
for idx, score, doc in hybrid_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.4f} | {doc}")

print("\nThree way hybrid plus Jaccard:")
for idx, score, doc in three_way_hybrid(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.4f} | {doc}")


Query: 'adorable baby cat'

TFIDF only:
  [5] score=0.525 | Puppies and kittens are adorable baby animals
  [0] score=0.152 | The cat sat on the mat
  [1] score=0.122 | The dog chased the cat in the garden

Embedding only:
  [5] score=0.987 | Puppies and kittens are adorable baby animals
  [2] score=0.979 | A kitten is a young cat
  [7] score=0.779 | Cats and dogs are the most popular pets worldwide

Two way hybrid TFIDF plus embedding:
  [5] score=0.0328 | Puppies and kittens are adorable baby animals
  [2] score=0.0318 | A kitten is a young cat
  [0] score=0.0315 | The cat sat on the mat

Three way hybrid plus Jaccard:
  [5] score=0.0492 | Puppies and kittens are adorable baby animals
  [0] score=0.0476 | The cat sat on the mat
  [2] score=0.0476 | A kitten is a young cat


## What You Learned

You now know why retrieval is needed before generation.

You implemented six retrieval ideas from simple matching to hybrid fusion.

You can now move to the vector embedding lesson and connect this retrieval intuition to dense semantic representations.
